# 🤖 SAM2D + CCPP 演算法路徑規劃

本筆記本展示如何使用 CCPP 演算法在草莓田中進行路徑規劃，並結合 SAM2 產生的 2D 地圖。  
依據 **GEMINI.md** 規範與 CCPP 雙側側拍需求：
- 解析度固定為 **0.1m**。
- 核心演算法：**CCPP (Complete Coverage Path Planning)**。
- 運動約束：**CCPP 雙側相機全覆蓋 (Double-Sided Camera CCPP Zigzag Coverage)**，每行依序 zigzag 遍歷且不跳行。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# 隔離 Jupyter Kernel 參數
sys.argv = [sys.argv[0]]

from ccpp_planner import CCPPPlanner

print("🚀 環境初始化完成")

## 1. 載入地圖數據
載入由 SAM2 分割出的 2D 網格地圖 (`farm_grid_map.csv`)。

In [ ]:
MAP_PATH = 'farm_grid_map.csv'
RESOLUTION = 0.1 # 0.1m per grid

if os.path.exists(MAP_PATH):
    grid_map = pd.read_csv(MAP_PATH, header=None).values
    print(f"✅ 成功載入地圖: {grid_map.shape} (約 {grid_map.shape[0]*RESOLUTION:.1f}m x {grid_map.shape[1]*RESOLUTION:.1f}m)")
else:
    # 如果不存在，建立一個模擬地圖 (11m x 19.5m)
    rows, cols = 110, 195
    grid_map = np.ones((rows, cols))
    # 模擬壟 (Obstacles for path planning if we want to stay in trenches)
    # 壟寬 0.7m, 週期 1.2m
    for c in range(cols):
        x = c * RESOLUTION
        if (x % 1.2) < 0.7:
            grid_map[:, c] = 0 # 壟部視為障礙物 (自走車行走於溝渠)
    print("⚠️ 使用模擬地圖參數生成")

plt.figure(figsize=(12, 6))
plt.imshow(grid_map, cmap='gray', origin='lower')
plt.title("Farm Grid Map (White=Trench, Black=Ridge)")
plt.show()

## 2. A* 路徑規劃實作
測試點對點導航。

In [ ]:
planner = CCPPPlanner(grid_map, resolution=RESOLUTION)

# 定義起點與終點 (Meters)
start_point = (1.0, 1.0) # (y, x)
goal_point = (9.0, 10.0)

path = planner.plan(start_point, goal_point)

if path:
    print(f"✅ 找到路徑！總長度: {len(path)*RESOLUTION:.2f} 公尺")
    path_np = np.array(path)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(grid_map, cmap='gray', origin='lower', extent=[0, grid_map.shape[1]*0.1, 0, grid_map.shape[0]*0.1])
    plt.plot(path_np[:, 1], path_np[:, 0], 'r-', linewidth=2, label='CCPP Path')
    plt.scatter(start_point[1], start_point[0], c='g', label='Start')
    plt.scatter(goal_point[1], goal_point[0], c='b', label='Goal')
    plt.legend()
    plt.title("CCPP Path Planning Results")
    plt.xlabel("X (meters)")
    plt.ylabel("Y (meters)")
    plt.show()
else:
    print("❌ 找不到路徑，請檢查起點終點是否在障礙物內")

## 3. CCPP 雙側相機全覆蓋路徑規劃 (CCPP Double-Sided Camera Zigzag Coverage)
生成覆蓋全田的 zigzag 巡檢路徑點（不跳行，每條溝渠依序全覆蓋）。

In [ ]:
def get_ccpp_coverage_path(grid_map, resolution=0.1):
    """
    實施 CCPP 雙側相機全覆蓋路徑規劃（無跳行，每一行依序 Zigzag 全覆蓋）
    """
    rows, cols = grid_map.shape
    # 尋找所有溝渠 (Value=1) 的中心線
    # 假設溝渠寬度約 0.5m (5 cells)
    trench_centers = []
    period_cells = int(1.2 / resolution)
    for c in range(0, cols, period_cells):
        # 找到壟與壟之間的中心點
        center_x = c + int(0.95 / resolution) # 偏移到溝渠中心
        if center_x < cols:
            trench_centers.append(center_x)
    
    # CCPP 順序: 每行依序走過，不跳行 (0, 1, 2, 3, 4...)
    order = trench_centers
    
    full_path = []
    current_pos = (0.5, order[0] * resolution)
    
    for i, tx in enumerate(order):
        tx_m = tx * resolution
        # 走道起點與終點 (交替方向以實現 zigzag)
        if i % 2 == 0:
            p1 = (1.0, tx_m)
            p2 = (rows * resolution - 1.0, tx_m)
        else:
            p1 = (rows * resolution - 1.0, tx_m)
            p2 = (1.0, tx_m)
        
        # 使用 CCPP 路徑規劃連接到走道起點 (如果不是第一條)
        segment = planner.plan(current_pos, p1)
        if segment: 
            full_path.extend(segment)
        
        # 直線掃描走道 (點集)
        y_steps = np.linspace(p1[0], p2[0], 20)
        for y in y_steps:
            full_path.append((y, tx_m))
            
        current_pos = p2
        
    return full_path

# 保留別名以防外部調用錯誤
get_skip_row_path = get_ccpp_coverage_path

coverage_path = get_ccpp_coverage_path(grid_map)
cp_np = np.array(coverage_path)

plt.figure(figsize=(15, 8))
plt.imshow(grid_map, cmap='gray', origin='lower', extent=[0, grid_map.shape[1]*0.1, 0, grid_map.shape[0]*0.1], alpha=0.3)
plt.plot(cp_np[:, 1], cp_np[:, 0], 'r-', linewidth=1, alpha=0.8)
plt.scatter(cp_np[::10, 1], cp_np[::10, 0], c='b', s=5, label='Waypoints')
plt.title("CCPP Double-Sided Camera Zigzag Coverage Path")
plt.legend()
plt.show()

print(f"📊 規劃完成，總路徑點數: {len(coverage_path)}")